# 03 — Skill Extraction: TF-IDF vs KeyBERT

**Purpose:** Extract skills/competencies from curriculum descriptions and job postings using two methods (TF-IDF keyword extraction and KeyBERT), then compute alignment metrics.

**Inputs:**
- `data/processed/university/final_curriculum_dataset.csv` — 1,161 curriculum rows (all with English course names)
- `data/processed/jobs/final_jobs_dataset_it_only.csv` — 1,068 deduplicated job postings

**Outputs:** `data/processed/skills/` — per-method skill lists and comparison CSV

**Note on reproducibility:** This notebook sets `random.seed(42)` and `np.random.seed(42)` before KeyBERT extraction to maximize reproducibility. Minor variations in KeyBERT MMR output may still occur across different hardware/library versions. If re-running, re-run notebooks 05 and 06 afterward to keep the pipeline consistent.

In [1]:
import pandas as pd
import numpy as np
import json
import re
import random
import warnings
from pathlib import Path
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer
from keybert import KeyBERT

import os
# Set working directory to project root regardless of launch location
_nb_path = globals().get("__vsc_ipynb_file__") or globals().get("__file__", None)
if _nb_path:
    # Launched from VS Code or as script — go up from notebooks/3_analysis/
    os.chdir(Path(_nb_path).resolve().parent.parent.parent)
elif not (Path.cwd() / "data").exists():
    # Launched from wrong cwd — try to find project root
    _root = Path.cwd()
    for _ in range(4):
        if (_root / "data").exists():
            break
        _root = _root.parent
    os.chdir(_root)
assert (Path.cwd() / "data").exists(), f"Cannot find project root from {Path.cwd()}"

# Fix random seeds for reproducibility (KeyBERT MMR uses numpy random internally)
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

warnings.filterwarnings('ignore')

OUTPUT_DIR = Path('data/processed/skills')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TOP_N = 10  # keywords per document
print("Setup complete.")

/Users/lianaaghamalyan/PycharmProjects/Dedupe/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup complete.


## 1. Load and Prepare Data

In [2]:
# Curriculum: use final_curriculum_dataset.csv (has course_name + description for all 1,161 rows)
curriculum = pd.read_csv('data/processed/university/final_curriculum_dataset.csv')

# Jobs
jobs = pd.read_csv('data/processed/jobs/final_jobs_dataset_it_only.csv')

print(f'Curriculum: {len(curriculum)} rows')
print(f'Jobs: {len(jobs)} rows')
print(f'Curriculum with description: {curriculum["description"].notna().sum()}')
print(f'Jobs with full_text: {jobs["full_text"].notna().sum()}')

Curriculum: 1161 rows
Jobs: 753 rows
Curriculum with description: 933
Jobs with full_text: 753


In [ ]:
# Build text for curriculum: NAMES ONLY for fair cross-university comparison
# Rationale: YSU (691 courses) and AUA (242 courses) have descriptions averaging
# 272 and 543 chars respectively, while NUACA, RAU, and UFAR have ZERO descriptions.
# Using names+descriptions would bias alignment scores by data availability, not
# curriculum quality. Sensitivity analysis (notebook 03) quantifies the description
# impact separately using AUA Variant A (names-only) vs Variant B (names+descriptions).
curriculum['_text'] = curriculum['course_name'].fillna('').str.strip()

# Jobs: use full_text directly
jobs['_text'] = jobs['full_text'].fillna('')

print(f'Curriculum _text coverage: {(curriculum["_text"].str.len() > 5).sum()}/{len(curriculum)}')
print(f'Jobs _text coverage: {(jobs["_text"].str.len() > 5).sum()}/{len(jobs)}')

# Show per-university text stats
for u in curriculum['university'].unique():
    sub = curriculum[curriculum['university'] == u]
    avg_len = sub['_text'].str.len().mean()
    has_text = (sub['_text'].str.len() > 5).sum()
    print(f'  {u[:45]:45} {has_text:4}/{len(sub):4} docs, avg {avg_len:.0f} chars')

print(f'\nCurriculum _text sample:\n{curriculum["_text"].iloc[0][:200]}')
print(f'\nJobs _text sample:\n{jobs["_text"].iloc[0][:200]}')

## 2. Text Preprocessing

In [4]:
# --- Boilerplate removal for jobs ---
# Detect paragraphs that appear in 4+ job postings (company 'About Us' sections, etc.)

para_counter = Counter()
for text in jobs['_text']:
    paras = [p.strip() for p in re.split(r'\n\s*\n|\r\n\s*\r\n', str(text)) if len(p.strip()) > 80]
    for p in paras:
        para_counter[p] += 1

boilerplate_paras = {p for p, count in para_counter.items() if count >= 4}
print(f'Found {len(boilerplate_paras)} boilerplate paragraphs (appearing in 4+ jobs)')

# Show a few examples
for i, bp in enumerate(list(boilerplate_paras)[:3]):
    print(f'\n--- Boilerplate #{i+1} (first 120 chars) ---')
    print(bp[:120] + '...')

def remove_boilerplate(text, boilerplate_set):
    paras = re.split(r'\n\s*\n|\r\n\s*\r\n', str(text))
    cleaned = [p for p in paras if p.strip() not in boilerplate_set]
    return '\n\n'.join(cleaned)

jobs['_text'] = jobs['_text'].apply(lambda t: remove_boilerplate(t, boilerplate_paras))
print(f'\nBoilerplate removed from jobs _text.')

Found 85 boilerplate paragraphs (appearing in 4+ jobs)

--- Boilerplate #1 (first 120 chars) ---
Miro handles and uses personal data of job applicants in line with its Recruitment Privacy Policy found here....

--- Boilerplate #2 (first 120 chars) ---
Workato transforms technology complexity into business opportunity. As the leader in enterprise orchestration, Workato h...

--- Boilerplate #3 (first 120 chars) ---
BrainRocket is a global company creating end-to-end tech products for clients across Fintech, iGaming, and Marketing. ‍Y...

Boilerplate removed from jobs _text.


In [5]:
# --- Expanded domain stopwords ---
# Words that pass sklearn's English stopword filter but are NOT skills

CUSTOM_STOPWORDS = {
    # Academic verbs & filler
    'familiarize', 'familiarization', 'introduce', 'introduction', 'introducing',
    'study', 'studies', 'studying', 'learn', 'learning', 'learned',
    'understand', 'understanding', 'knowledge', 'concepts', 'concept',
    'course', 'courses', 'class', 'classes', 'lecture', 'lectures',
    'instructor', 'student', 'students', 'semester', 'academic',
    'discuss', 'discussion', 'discussions', 'examine', 'explore', 'exploring',
    'review', 'overview', 'provide', 'provides', 'providing', 'provided',
    'develop', 'developing', 'development', 'include', 'includes', 'including',
    'cover', 'covers', 'covering', 'covered', 'focus', 'focuses', 'focused',
    'present', 'presents', 'presented', 'presentation',
    'teach', 'teaching', 'taught', 'educate', 'education',
    'assignment', 'assignments', 'exam', 'exams', 'examination', 'test', 'tests',
    'grade', 'grading', 'credit', 'credits', 'hours', 'hour', 'time', 'week', 'weeks',
    'topic', 'topics', 'subject', 'subjects', 'material', 'materials',
    'practice', 'practical', 'theoretical', 'theory', 'principles', 'principle',
    'basic', 'basics', 'advanced', 'fundamental', 'fundamentals',
    'method', 'methods', 'approach', 'approaches', 'technique', 'techniques',
    'problem', 'problems', 'solution', 'solutions', 'solve', 'solving',
    'example', 'examples', 'exercise', 'exercises',
    # Job posting filler
    'seeking', 'looking', 'hiring', 'interview', 'apply', 'application',
    'candidate', 'candidates', 'applicant', 'applicants',
    'experience', 'experienced', 'years', 'year', 'proven', 'strong',
    'ability', 'able', 'capable', 'excellent', 'good', 'great', 'best',
    'work', 'working', 'job', 'position', 'role', 'responsibilities', 'responsibility',
    'responsible', 'require', 'required', 'requirements', 'requirement',
    'prefer', 'preferred', 'plus', 'bonus', 'nice',
    'team', 'teams', 'collaborate', 'collaboration', 'collaborative',
    'closely', 'independently', 'environment', 'fast', 'paced',
    'company', 'organization', 'business', 'industry', 'professional',
    'opportunity', 'opportunities', 'growth', 'career',
    'benefit', 'benefits', 'salary', 'compensation', 'package',
    'office', 'remote', 'hybrid', 'location', 'based',
    'comply', 'compliance', 'labor', 'equal', 'employer',
    'video', 'submit', 'resume', 'cv', 'letter',
    'day', 'days', 'month', 'months', 'full', 'part',
    'join', 'offer', 'offers', 'offered',
    'communicate', 'communication', 'communications', 'written', 'verbal',
    'manage', 'management', 'managing', 'manager',
    'lead', 'leading', 'leader', 'leadership',
    'support', 'supporting', 'ensure', 'ensuring',
    'create', 'creating', 'build', 'building', 'built',
    'implement', 'implementing', 'implementation',
    'maintain', 'maintaining', 'maintenance',
    'use', 'using', 'used', 'utilize', 'utilizing',
    'new', 'current', 'existing', 'various', 'multiple', 'different',
    'high', 'level', 'quality', 'standard', 'standards',
    'process', 'processes', 'project', 'projects',
    'need', 'needs', 'needed', 'want', 'help', 'must',
    'll', 've', 'don', 'didn', 'doesn', 'isn', 'aren', 'won',
    # Geography
    'yerevan', 'armenia', 'armenian', 'russia', 'russian',
    # Generic
    'also', 'well', 'make', 'take', 'get', 'set', 'way',
    'like', 'based', 'related', 'relevant', 'key', 'important',
    'first', 'second', 'third', 'one', 'two', 'three',
    'part', 'end', 'start', 'beginning',
    'description', 'title', 'name', 'type', 'field',
    'information', 'system', 'systems',
}

print(f'Custom stopwords: {len(CUSTOM_STOPWORDS)} terms')

Custom stopwords: 295 terms


In [6]:
# --- Company name token extraction ---
# Prevent company names from appearing as 'skills'

SKILL_WORDS = {
    'data', 'cloud', 'software', 'digital', 'tech', 'technology', 'smart',
    'security', 'ai', 'cyber', 'analytics', 'web', 'mobile', 'network',
    'design', 'code', 'logic', 'compute', 'micro', 'vision', 'quantum',
}

company_tokens = set()
for name in jobs['company_name'].dropna().unique():
    tokens = set(re.findall(r'[a-zA-Z]{3,}', name.lower()))
    tokens -= SKILL_WORDS
    company_tokens.update(tokens)

print(f'Company name tokens to filter: {len(company_tokens)}')
print(f'Sample: {sorted(list(company_tokens))[:20]}')

Company name tokens to filter: 239
Sample: ['abs', 'acba', 'acra', 'acro', 'actual', 'addevice', 'aderant', 'adobe', 'agency', 'alango', 'alex', 'align', 'altamira', 'and', 'anti', 'aobyte', 'apartment', 'ardshinbank', 'areg', 'ark']


In [7]:
# --- Skill-likeness post-filter ---

GENERIC_UNIGRAMS = {
    # --- Original set ---
    'ability', 'able', 'according', 'across', 'addition', 'also',
    'area', 'areas', 'base', 'best', 'case', 'cases', 'change',
    'changes', 'common', 'complete', 'complex', 'continuous',
    'cross', 'currently', 'customer', 'customers', 'deep',
    'defined', 'detail', 'details', 'different', 'end',
    'ensure', 'entire', 'every', 'expected', 'experience',
    'following', 'general', 'given', 'global', 'goals',
    'group', 'groups', 'hands', 'help', 'identify', 'impact',
    'internal', 'international', 'issue', 'issues', 'item',
    'items', 'large', 'latest', 'least', 'line', 'long',
    'main', 'major', 'meet', 'member', 'members', 'mind',
    'modern', 'next', 'note', 'number',
    'open', 'order', 'others', 'overall', 'people', 'place',
    'plan', 'plans', 'point', 'possible', 'product', 'products',
    'real', 'record', 'records', 'result', 'results', 'right',
    'run', 'running', 'scale', 'service', 'services', 'side',
    'similar', 'simple', 'small', 'source', 'space', 'special',
    'specific', 'stage', 'state', 'step', 'steps', 'success',
    'task', 'tasks', 'term', 'terms', 'thing', 'things',
    'tool', 'tools', 'total', 'track', 'turn', 'update',
    'user', 'users', 'value', 'values', 'view', 'world',
    # --- Expanded: overlap audit 2026-03-23 (generic words misidentified as skills) ---
    'access', 'achieve', 'acquired', 'acquisition', 'action', 'active',
    'activities', 'activity', 'additionally', 'address', 'aimed', 'alongside',
    'appropriate', 'articulate', 'assessment', 'assessments', 'assist',
    'association', 'background', 'behavior', 'better', 'big', 'block', 'body',
    'capabilities', 'capture', 'care', 'causes', 'chain', 'challenges',
    'choice', 'choose', 'civil', 'clarify', 'clear', 'client',
    'collecting', 'collection', 'companies', 'competencies', 'completion',
    'components', 'comprehensive', 'conducted', 'confidence', 'connection',
    'considerations', 'considered', 'consistency', 'consumer', 'content',
    'continuity', 'coordinate', 'core', 'corporate', 'cost', 'coverage',
    'creation', 'creators', 'criteria', 'critical', 'culture', 'curve',
    'cutting', 'daily', 'deal', 'decision', 'decisions', 'demand',
    'demonstrations', 'depth', 'designed', 'designers', 'designs', 'detailed',
    'developed', 'device', 'devices', 'diffusion', 'direction', 'directions',
    'discover', 'distribution', 'does', 'domains', 'driven',
    'economic', 'educational', 'effective', 'effectiveness', 'effects',
    'efficiency', 'efficient', 'elements', 'emergency', 'emphasizing',
    'employment', 'enable', 'enabling', 'encountered', 'engage', 'engineers',
    'english', 'environments', 'equipment', 'error', 'errors', 'estimates',
    'evaluate', 'evaluation', 'events', 'expand', 'experiments', 'explain',
    'exploration', 'exposure', 'external', 'extract',
    'facilities', 'facing', 'failures', 'familiar', 'familiarity', 'family',
    'files', 'final', 'findings', 'fit', 'flexibility', 'flow', 'flows',
    'follow', 'foreign', 'form', 'formal', 'formats', 'forms', 'foster',
    'fostering', 'foundation', 'foundations', 'free', 'french', 'function',
    'functional', 'functionality', 'functioning', 'functions', 'future', 'gained',
    'generation', 'german', 'globally', 'goal', 'grammar', 'guest', 'guide',
    'handling', 'hard', 'healthy', 'higher', 'history', 'human', 'hypotheses',
    'idea', 'ideas', 'identifying', 'image', 'images', 'implementations',
    'improvement', 'independent', 'individuals', 'innovation', 'innovative',
    'insights', 'integral', 'integrate', 'intelligent', 'interaction',
    'interactions', 'intermediate', 'internship', 'investigation', 'investment',
    'lab', 'laboratory', 'language', 'languages', 'layer', 'legal', 'lifecycle',
    'limitations', 'limits', 'lines', 'lives', 'local', 'look', 'loop',
    'making', 'managers', 'manual', 'market', 'master', 'mean',
    'measurable', 'measure', 'measures', 'measuring', 'mechanisms',
    'methodologies', 'methodology', 'minimum', 'mixed', 'module', 'momentum',
    'money', 'motion', 'moving', 'music', 'natural', 'necessary', 'non',
    'numbers', 'object', 'objects', 'opening', 'operation', 'operational',
    'operations', 'options', 'organic', 'organizational', 'outcome', 'outcomes',
    'packages', 'paradigms', 'partial', 'participate', 'parts', 'perform',
    'performance', 'perspectives', 'phase', 'physical', 'policies', 'portfolio',
    'post', 'potential', 'power', 'pre', 'prepare', 'procedures',
    'production', 'professionals', 'program', 'programs', 'promote', 'proof',
    'prospects', 'public', 'pure', 'range', 'rapid', 'realistic',
    'receive', 'reducing', 'regarding', 'region', 'relations',
    'relationship', 'relationships', 'report', 'reports',
    'resource', 'responses', 'risk', 'risks', 'scenarios',
    'schemes', 'school', 'sciences', 'scientific', 'scientists', 'scope',
    'search', 'sector', 'self', 'semi', 'series', 'sessions',
    'sets', 'shaping', 'shared', 'skills', 'solar', 'solid', 'sound',
    'speaking', 'specifically', 'specifications',
    'stable', 'statements', 'strategies', 'strategy', 'structure',
    'structured', 'structures', 'sub', 'succeed', 'successful', 'successfully',
    'sufficient', 'supply', 'sustainability', 'tailored', 'taken', 'target',
    'technological', 'thinking', 'tomorrow', 'transactions', 'transfer',
    'transformation', 'trends', 'turkish', 'types', 'uncertainty', 'units',
    'usage', 'useful', 'validation', 'variance', 'variety', 'views',
    'ways', 'white', 'wide', 'word', 'writing',
}

# Multi-word phrases that are NOT skills (slip through unigram filter)
NOISE_PHRASES = {
    'cutting edge', 'wide range', 'decision making', 'web mobile',
    'design data', 'machine models', 'data data', 'technologies software',
    'testing software', 'hardware software', 'stability performance',
}

def is_skill_like(keyword, company_tokens_set, custom_stops):
    kw = keyword.lower().strip()
    tokens = kw.split()
    if len(kw) < 3:
        return False
    if re.match(r'^[\d\s.,%]+$', kw):
        return False
    if kw in NOISE_PHRASES:
        return False
    if all(t in company_tokens_set for t in tokens):
        return False
    if len(tokens) == 1:
        if kw in custom_stops or kw in GENERIC_UNIGRAMS:
            return False
    stop_count = sum(1 for t in tokens if t in custom_stops or t in GENERIC_UNIGRAMS)
    if len(tokens) >= 2 and stop_count >= len(tokens):
        return False
    if len(tokens) >= 2 and stop_count / len(tokens) > 0.6:
        return False
    return True

# Quick test
test_kws = ['python', 'machine learning', 'familiarize students', 'xometry nasdaq xmtr',
            'sql', 'data analysis', 'seeking experienced', '123', 'ci cd pipeline']
for kw in test_kws:
    result = is_skill_like(kw, company_tokens, CUSTOM_STOPWORDS)
    print(f"  '{kw}' -> {result}")

  'python' -> True
  'machine learning' -> True
  'familiarize students' -> False
  'xometry nasdaq xmtr' -> True
  'sql' -> True
  'data analysis' -> True
  'seeking experienced' -> False
  '123' -> False
  'ci cd pipeline' -> True


## 3. TF-IDF Keyword Extraction

In [8]:
def extract_tfidf_keywords(texts, doc_ids, top_n=TOP_N, label=''):
    from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
    all_stops = list(ENGLISH_STOP_WORDS | CUSTOM_STOPWORDS)
    
    # token_pattern note: allows +, #, . to capture C++, C#, .NET-style terms.
    # LIMITATION: tokens must start with [a-zA-Z], so '.NET' (starts with '.') is
    # still dropped. Likewise, single/two-char tokens ('C', 'Go', 'R') fall below
    # min_df=2 or is_skill_like(len<3). These are recovered in notebook 06 via
    # the tech lexicon (match_lexicon), which uses regex directly on raw text.
    vectorizer = TfidfVectorizer(
        ngram_range=(1, 3),
        max_df=0.85,
        min_df=2,       # terms in <2 docs discarded; see Part 4 of notebook 03 for sensitivity
        stop_words=all_stops,
        max_features=15000,
        token_pattern=r'(?u)\b[a-zA-Z][a-zA-Z+#.]{1,}\b',
    )
    
    tfidf_matrix = vectorizer.fit_transform(texts)
    feature_names = vectorizer.get_feature_names_out()
    
    results = {}
    for i in range(len(texts)):
        row = tfidf_matrix[i].toarray().flatten()
        top_indices = row.argsort()[-top_n*2:][::-1]
        
        keywords = []
        for idx in top_indices:
            if row[idx] > 0:
                kw = feature_names[idx]
                if is_skill_like(kw, company_tokens, CUSTOM_STOPWORDS):
                    keywords.append((kw, float(round(row[idx], 4))))
            if len(keywords) >= top_n:
                break
        
        results[str(doc_ids[i])] = keywords
        
        if (i + 1) % 200 == 0:
            print(f'  [{label}] Processed {i+1}/{len(texts)} docs')
    
    print(f'  [{label}] Done: {len(texts)} docs, avg {np.mean([len(v) for v in results.values()]):.1f} keywords/doc')
    return results

print('TF-IDF extractor ready.')

TF-IDF extractor ready.


In [9]:
# Filter to docs with text
curr_mask = curriculum['_text'].str.len() > 10
jobs_mask = jobs['_text'].str.len() > 10

print(f'Curriculum docs with text: {curr_mask.sum()} / {len(curriculum)}')
print(f'Jobs docs with text: {jobs_mask.sum()} / {len(jobs)}')

# Log excluded courses explicitly
excluded_courses = curriculum.loc[~curr_mask, ['course_id', 'university', 'course_name']].copy()
if len(excluded_courses) > 0:
    print(f'\n⚠ {len(excluded_courses)} courses excluded (text ≤ 10 chars):')
    for _, row in excluded_courses.iterrows():
        print(f'  course_id={row["course_id"]}  {row["university"]}  {row["course_name"][:60]}')
    excluded_courses.to_csv('data/processed/skills/excluded_courses.csv', index=False)
    print(f'  → Saved to data/processed/skills/excluded_courses.csv')

print('\n--- TF-IDF: Curriculum ---')
tfidf_curriculum = extract_tfidf_keywords(
    curriculum.loc[curr_mask, '_text'].tolist(),
    curriculum.loc[curr_mask, 'course_id'].tolist(),
    label='curriculum'
)

print('\n--- TF-IDF: Jobs ---')
tfidf_jobs = extract_tfidf_keywords(
    jobs.loc[jobs_mask, '_text'].tolist(),
    jobs.loc[jobs_mask].index.tolist(),
    label='jobs'
)

print(f'\nTF-IDF curriculum skills: {len(tfidf_curriculum)} docs')
print(f'TF-IDF jobs skills: {len(tfidf_jobs)} docs')

Curriculum docs with text: 1133 / 1161
Jobs docs with text: 697 / 753

⚠ 28 courses excluded (text ≤ 10 chars):
  course_id=829  American University of Armenia  Capstone 1
  course_id=952  National University of Architecture and Construction of Armenia  WEB GIS
  course_id=954  National University of Architecture and Construction of Armenia  Practice
  course_id=971  National University of Architecture and Construction of Armenia  Practice
  course_id=977  National University of Architecture and Construction of Armenia  Russian
  course_id=979  National University of Architecture and Construction of Armenia  Physics
  course_id=990  National University of Architecture and Construction of Armenia  Physics
  course_id=1021  National University of Architecture and Construction of Armenia  Management
  course_id=1028  National University of Architecture and Construction of Armenia  Philosophy
  course_id=1029  National University of Architecture and Construction of Armenia  Management
  co

In [10]:
# Preview TF-IDF results
print('=== TF-IDF Curriculum - first 5 docs ===')
for doc_id, kws in list(tfidf_curriculum.items())[:5]:
    row = curriculum[curriculum['course_id'] == doc_id]
    name = row['course_name'].iloc[0] if len(row) > 0 else doc_id
    skills = [kw for kw, score in kws]
    print(f'  {name}: {skills}')

print('\n=== TF-IDF Jobs - first 5 docs ===')
for doc_id, kws in list(tfidf_jobs.items())[:5]:
    idx = int(doc_id)
    title = jobs.loc[idx, 'job_title'] if idx in jobs.index else doc_id
    skills = [kw for kw, score in kws]
    print(f'  {title}: {skills}')

=== TF-IDF Curriculum - first 5 docs ===
  1: ['python', 'data', 'language data variables', 'language data', 'arrays functions', 'international practices', 'variables arrays functions', 'variables arrays', 'python python programming', 'python python']
  2: ['strategic', 'planning']
  3: ['mathematical', 'modeling applications', 'economic mathematical', 'main extremal', 'optimization main extremal', 'applications mathematical', 'mathematical optimization', 'optimization main', 'extremal', 'mathematical optimization main']
  4: ['probabilistic', 'continuity derivatives', 'probability data', 'derivatives applications', 'applications definite', 'data scientists', 'probabilistic models', 'mindset', 'probabilistic mindset', 'elementary']
  5: ['visualization', 'data visualization data', 'analysis visualization', 'visualization data', 'skill', 'data analysis visualization', 'data', 'data visualization', 'data analysis', 'analysis']

=== TF-IDF Jobs - first 5 docs ===
  Application Support Eng

## 4. KeyBERT Keyword Extraction

In [11]:
# Initialize KeyBERT with lightweight model
kw_model = KeyBERT(model='all-MiniLM-L6-v2')
print('KeyBERT model loaded (all-MiniLM-L6-v2, 22M params)')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10642.56it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


KeyBERT model loaded (all-MiniLM-L6-v2, 22M params)


In [12]:
def extract_keybert_keywords(texts, doc_ids, top_n=TOP_N, label=''):
    results = {}
    extract_n = top_n + 5
    
    for i, text in enumerate(texts):
        if len(text.strip()) < 20:
            results[str(doc_ids[i])] = []
            continue
        
        try:
            raw_kws = kw_model.extract_keywords(
                text,
                keyphrase_ngram_range=(1, 3),
                stop_words='english',
                top_n=extract_n,
                use_mmr=True,
                diversity=0.5,
            )
        except Exception:
            results[str(doc_ids[i])] = []
            continue
        
        filtered = []
        for kw, score in raw_kws:
            if is_skill_like(kw, company_tokens, CUSTOM_STOPWORDS):
                filtered.append((kw, float(round(score, 4))))
            if len(filtered) >= top_n:
                break
        
        results[str(doc_ids[i])] = filtered
        
        if (i + 1) % 100 == 0:
            print(f'  [{label}] Processed {i+1}/{len(texts)} docs')
    
    print(f'  [{label}] Done: {len(texts)} docs, avg {np.mean([len(v) for v in results.values()]):.1f} keywords/doc')
    return results

print('KeyBERT extractor ready.')

KeyBERT extractor ready.


In [13]:
print('--- KeyBERT: Curriculum ---')
keybert_curriculum = extract_keybert_keywords(
    curriculum.loc[curr_mask, '_text'].tolist(),
    curriculum.loc[curr_mask, 'course_id'].tolist(),
    label='curriculum'
)

print('\n--- KeyBERT: Jobs ---')
keybert_jobs = extract_keybert_keywords(
    jobs.loc[jobs_mask, '_text'].tolist(),
    jobs.loc[jobs_mask].index.tolist(),
    label='jobs'
)

print(f'\nKeyBERT curriculum skills: {len(keybert_curriculum)} docs')
print(f'KeyBERT jobs skills: {len(keybert_jobs)} docs')

--- KeyBERT: Curriculum ---
  [curriculum] Processed 100/1133 docs
  [curriculum] Processed 200/1133 docs
  [curriculum] Processed 300/1133 docs
  [curriculum] Processed 400/1133 docs
  [curriculum] Processed 500/1133 docs
  [curriculum] Processed 600/1133 docs
  [curriculum] Processed 700/1133 docs
  [curriculum] Processed 800/1133 docs
  [curriculum] Processed 900/1133 docs
  [curriculum] Processed 1000/1133 docs
  [curriculum] Done: 1133 docs, avg 6.6 keywords/doc

--- KeyBERT: Jobs ---
  [jobs] Processed 100/697 docs
  [jobs] Processed 200/697 docs
  [jobs] Processed 300/697 docs
  [jobs] Processed 400/697 docs
  [jobs] Processed 500/697 docs
  [jobs] Processed 600/697 docs
  [jobs] Done: 697 docs, avg 9.3 keywords/doc

KeyBERT curriculum skills: 1133 docs
KeyBERT jobs skills: 697 docs


In [14]:
# Preview KeyBERT results
print('=== KeyBERT Curriculum - first 5 docs ===')
for doc_id, kws in list(keybert_curriculum.items())[:5]:
    row = curriculum[curriculum['course_id'] == doc_id]
    name = row['course_name'].iloc[0] if len(row) > 0 else doc_id
    skills = [kw for kw, score in kws]
    print(f'  {name}: {skills}')

print('\n=== KeyBERT Jobs - first 5 docs ===')
for doc_id, kws in list(keybert_jobs.items())[:5]:
    idx = int(doc_id)
    title = jobs.loc[idx, 'job_title'] if idx in jobs.index else doc_id
    skills = [kw for kw, score in kws]
    print(f'  {title}: {skills}')

=== KeyBERT Curriculum - first 5 docs ===
  1: ['data visualization python', 'visualization python teaching', 'python teaching', 'fundamentals python', 'programming language', 'practices data', 'arrays functions']
  2: ['methods strategic planning', 'strategic planning develop', 'strategic planning', 'strategic planning enable', 'knowledge strategic planning', 'planning develop', 'strategic']
  3: ['economic mathematical modeling', 'mathematical modeling applications', 'mathematical theory optimization', 'mathematical methods', 'methods calculations data', 'mathematical', 'data analysis english', 'modeling applications teach', 'extremal']
  4: ['probabilistic models concepts', 'mathematics probability theory', 'probability theory data', 'probabilistic mindset help', 'applied statistics', 'mathematics', 'integrals', 'continuity derivatives applications', 'functions elementary', 'applications definite indefinite']
  5: ['processes data visualization', 'data analysis visualization', 'data

## 5. Alignment Metrics

In [15]:
def compute_alignment(curriculum_skills, jobs_skills, method_name):
    curr_set = set()
    for kws in curriculum_skills.values():
        for kw, score in kws:
            curr_set.add(kw.lower().strip())
    
    jobs_set = set()
    for kws in jobs_skills.values():
        for kw, score in kws:
            jobs_set.add(kw.lower().strip())
    
    overlap = curr_set & jobs_set
    gap = jobs_set - curr_set
    surplus = curr_set - jobs_set
    
    coverage = len(overlap) / len(jobs_set) if jobs_set else 0
    jaccard = len(overlap) / len(curr_set | jobs_set) if (curr_set | jobs_set) else 0
    
    print(f'\n=== {method_name} Alignment ===')
    print(f'Curriculum unique skills: {len(curr_set)}')
    print(f'Job market unique skills: {len(jobs_set)}')
    print(f'Overlap: {len(overlap)}')
    print(f'Coverage rate (overlap/jobs): {coverage:.3f} ({coverage*100:.1f}%)')
    print(f'Jaccard similarity: {jaccard:.3f}')
    print(f'Gap (jobs only): {len(gap)}')
    print(f'Surplus (curriculum only): {len(surplus)}')
    print(f'\nTop 30 overlapping skills: {sorted(overlap)[:30]}')
    print(f'\nTop 30 gap skills (market needs, not taught): {sorted(gap)[:30]}')
    
    return {
        'method': method_name,
        'curriculum_unique': len(curr_set),
        'jobs_unique': len(jobs_set),
        'overlap': len(overlap),
        'coverage_rate': round(coverage, 4),
        'jaccard': round(jaccard, 4),
        'gap_count': len(gap),
        'surplus_count': len(surplus),
        'overlap_skills': sorted(overlap),
        'gap_skills': sorted(gap),
        'surplus_skills': sorted(surplus),
    }

print('Alignment function ready.')

Alignment function ready.


In [16]:
tfidf_alignment = compute_alignment(tfidf_curriculum, tfidf_jobs, 'TF-IDF')
keybert_alignment = compute_alignment(keybert_curriculum, keybert_jobs, 'KeyBERT')


=== TF-IDF Alignment ===
Curriculum unique skills: 3442
Job market unique skills: 3153
Overlap: 279
Coverage rate (overlap/jobs): 0.088 (8.8%)
Jaccard similarity: 0.044
Gap (jobs only): 2874
Surplus (curriculum only): 3163

Top 30 overlapping skills: ['accounting', 'addressing', 'agent', 'agents', 'agile', 'algorithm', 'algorithmic', 'algorithms', 'alignment', 'american', 'analog', 'analysis', 'analysis tools', 'analytical', 'analytics', 'analyze', 'analyzing', 'angular', 'annotation', 'applications', 'architecture', 'architecture modern', 'architectures', 'assembly', 'assessing', 'assurance', 'audio', 'automated', 'automation', 'big data']

Top 30 gap skills (market needs, not taught): ['a.m', 'a.m est', 'abap', 'abap objects', 'accelerator', 'acceptance', 'acceptance criteria', 'access control', 'access patterns', 'accessibility', 'accordance', 'according technical', 'account', 'accountability', 'accounts', 'accuracy', 'accuracy consistency', 'accurate', 'act', 'act highest', 'actio

## 6. Method Comparison

In [17]:
comparison = pd.DataFrame([
    {'Metric': 'Curriculum unique skills', 'TF-IDF': tfidf_alignment['curriculum_unique'], 'KeyBERT': keybert_alignment['curriculum_unique']},
    {'Metric': 'Job unique skills', 'TF-IDF': tfidf_alignment['jobs_unique'], 'KeyBERT': keybert_alignment['jobs_unique']},
    {'Metric': 'Overlap', 'TF-IDF': tfidf_alignment['overlap'], 'KeyBERT': keybert_alignment['overlap']},
    {'Metric': 'Coverage rate', 'TF-IDF': tfidf_alignment['coverage_rate'], 'KeyBERT': keybert_alignment['coverage_rate']},
    {'Metric': 'Gap (jobs - curriculum)', 'TF-IDF': tfidf_alignment['gap_count'], 'KeyBERT': keybert_alignment['gap_count']},
    {'Metric': 'Surplus (curriculum - jobs)', 'TF-IDF': tfidf_alignment['surplus_count'], 'KeyBERT': keybert_alignment['surplus_count']},
    {'Metric': 'Jaccard similarity', 'TF-IDF': tfidf_alignment['jaccard'], 'KeyBERT': keybert_alignment['jaccard']},
])

print(comparison.to_string(index=False))

                     Metric    TF-IDF   KeyBERT
   Curriculum unique skills 3442.0000 4812.0000
          Job unique skills 3153.0000 5530.0000
                    Overlap  279.0000   18.0000
              Coverage rate    0.0885    0.0033
    Gap (jobs - curriculum) 2874.0000 5512.0000
Surplus (curriculum - jobs) 3163.0000 4794.0000
         Jaccard similarity    0.0442    0.0017


In [18]:
# Top skills by frequency across documents
def top_skills_by_frequency(skill_dict, top_n=30):
    counter = Counter()
    for kws in skill_dict.values():
        for kw, score in kws:
            counter[kw.lower().strip()] += 1
    return counter.most_common(top_n)

print('=== Top 30 TF-IDF Curriculum Skills (by doc frequency) ===')
for skill, count in top_skills_by_frequency(tfidf_curriculum):
    print(f'  {skill}: {count} docs')

print('\n=== Top 30 TF-IDF Job Skills (by doc frequency) ===')
for skill, count in top_skills_by_frequency(tfidf_jobs):
    print(f'  {skill}: {count} docs')

print('\n=== Top 30 KeyBERT Curriculum Skills (by doc frequency) ===')
for skill, count in top_skills_by_frequency(keybert_curriculum):
    print(f'  {skill}: {count} docs')

print('\n=== Top 30 KeyBERT Job Skills (by doc frequency) ===')
for skill, count in top_skills_by_frequency(keybert_jobs):
    print(f'  {skill}: {count} docs')

=== Top 30 TF-IDF Curriculum Skills (by doc frequency) ===
  data: 66 docs
  programming: 57 docs
  mathematical: 52 docs
  analysis: 52 docs
  design: 47 docs
  algorithms: 37 docs
  applications: 36 docs
  models: 35 docs
  computer: 35 docs
  science: 32 docs
  linguistic: 32 docs
  digital: 28 docs
  machine: 27 docs
  speech: 22 docs
  testing: 22 docs
  equations: 22 docs
  engineering: 21 docs
  characteristics: 21 docs
  mathematics: 21 docs
  processing: 21 docs
  statistics: 20 docs
  applied: 20 docs
  optimization: 19 docs
  object oriented: 19 docs
  software: 19 docs
  linear: 18 docs
  modeling: 18 docs
  computing: 17 docs
  abilities: 17 docs
  texts: 17 docs

=== Top 30 TF-IDF Job Skills (by doc frequency) ===
  data: 84 docs
  design: 42 docs
  technical: 39 docs
  testing: 39 docs
  backend: 38 docs
  automation: 36 docs
  cloud: 34 docs
  software: 33 docs
  net: 30 docs
  infrastructure: 29 docs
  security: 26 docs
  gyumri: 24 docs
  hardware: 22 docs
  azure: 22

## 7. Save Outputs

In [19]:
# Save skill extractions as JSON
with open(OUTPUT_DIR / 'tfidf_curriculum_skills.json', 'w') as f:
    json.dump(tfidf_curriculum, f, indent=2)

with open(OUTPUT_DIR / 'tfidf_jobs_skills.json', 'w') as f:
    json.dump(tfidf_jobs, f, indent=2)

with open(OUTPUT_DIR / 'keybert_curriculum_skills.json', 'w') as f:
    json.dump(keybert_curriculum, f, indent=2)

with open(OUTPUT_DIR / 'keybert_jobs_skills.json', 'w') as f:
    json.dump(keybert_jobs, f, indent=2)

# Save comparison CSV
comparison.to_csv(OUTPUT_DIR / 'method_comparison.csv', index=False)

# Save alignment details
alignment_details = {
    'tfidf': {k: v for k, v in tfidf_alignment.items() if k != 'method'},
    'keybert': {k: v for k, v in keybert_alignment.items() if k != 'method'},
}
with open(OUTPUT_DIR / 'alignment_details.json', 'w') as f:
    json.dump(alignment_details, f, indent=2)

print('All outputs saved to data/processed/skills/')
for fn in sorted(OUTPUT_DIR.glob('*')):
    print(f'  {fn.name} ({fn.stat().st_size / 1024:.0f} KB)')

All outputs saved to data/processed/skills/
  alignment_details.json (491 KB)
  excluded_courses.csv (2 KB)
  keybert_curriculum_skills.json (436 KB)
  keybert_jobs_skills.json (385 KB)
  method_comparison.csv (0 KB)
  tfidf_curriculum_skills.json (483 KB)
  tfidf_jobs_skills.json (328 KB)


## 8. Summary

**Extraction pipeline:**
1. Combined `course_name` + `description` for curriculum; `full_text` for jobs
2. Removed boilerplate paragraphs (appearing in 4+ jobs) from job texts
3. Applied expanded stopword list (~295 custom + ~350 generic unigrams: academic filler, job posting boilerplate, geography, generic English)
4. Filtered company name tokens from extracted keywords
5. Post-filtered with `is_skill_like()` to reject generics, pure numbers, and noise

**Two methods compared:**
- **TF-IDF** (sklearn): corpus-relative term importance, n-grams (1-3), min_df=2, max_df=0.85
- **KeyBERT** (all-MiniLM-L6-v2): semantic keyword extraction with MMR diversity (diversity=0.5)

**Next steps:**
- Normalize extracted skills to ESCO taxonomy (cosine similarity, threshold >= 0.75)
- Build skill x source co-occurrence matrix
- Per-program alignment analysis